In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
import pandas as pd



In [2]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db = os.getenv("DB")
port = os.getenv("PORT")
role = os.getenv("ROLE")
pw = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

In [6]:
# Define your column lists
cols_deals = ['updated_at', 'deleted_at', 'id', 'legacy_partner_id', 'title', 'description', 'creators_requirement', 'hash_tags', 'status', 'deal_value', 'go_live_at', 'live_until', 'deal_type',
              'images', 'accepts_international', 'accepted_countries', 'social_requirement_type_id', 'product_name', 'schedule_type', 'schedule_model', 'legacy_id', 'tags', 'gender', 'featured_image', 'company_id', 'partner_id']
cols_comp = ['applicants_applications_count', 'cancelled_applications_count', 'company_locations', 'completed_applications_count', 'content_types', 'deal_created_at', 'deal_deleted_at', 'deal_id', 'deal_tags', 'deal_updated_at',
             'first_application_at', 'last_application_at', 'live_since', 'main_image', 'min_social_media_followers', 'pending_applications_count', 'planned_applications_count', 'rejected_applications_count', 'total_company_locations']

# 1. Load Deals
query_deals = f"SELECT {', '.join(cols_deals)} FROM public.deals;"
df_deals = pd.read_sql(query_deals, engine)

# 2. Load Deals Computed with built-in date parsing
# We add deal_id::text as deal_id_str directly in the SQL string
# query_comp = f"""
#     SELECT {', '.join(cols_comp)}, deal_id::text AS deal_id_str 
#     FROM public.deals_computed;
# """

query_comp = f"""
    SELECT {', '.join(cols_comp)}
    FROM public.deals_computed;
"""

df_deals_comp = pd.read_sql(
    query_comp,
    engine,
    parse_dates=['first_application_at', 'last_application_at']
)

# Merge
df_deals = pd.merge(df_deals,
                    df_deals_comp,
                    left_on='id', right_on='deal_id', how='left')

del df_deals_comp
# Text takes up a lot of memory, can drop since it is not needed
df_deals.drop(columns=['description', 'creators_requirement', 'schedule_model', 'main_image', 'images', 'id'], inplace=True)

# df_deals = df_deals.add_suffix('_deals')





## Cast columns to saveable datatypes

In [8]:

df_deals["first_application_at"] = pd.to_datetime(
    df_deals["first_application_at"], errors='coerce')
df_deals["last_application_at"] = pd.to_datetime(
    df_deals["last_application_at"], errors='coerce')



df_deals['deal_id'] = df_deals['deal_id'].astype(str)
# df_deals['id'] = df_deals['id'].astype(str)
df_deals['company_id'] = df_deals['company_id'].astype(str)
df_deals['partner_id'] = df_deals['partner_id'].astype(str)

import json

def safe_json_dump(x):
    # Handle NaNs or None
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    # Convert list/dict to valid JSON string
    return json.dumps(x)

# Apply JSON serialization
df_deals['company_locations'] = df_deals['company_locations'].apply(safe_json_dump)

In [9]:
df_deals

,updated_at,deleted_at,legacy_partner_id,title,hash_tags,status,deal_value,go_live_at,live_until,deal_type,...,deal_tags,deal_updated_at,first_application_at,last_application_at,live_since,min_social_media_followers,pending_applications_count,planned_applications_count,rejected_applications_count,total_company_locations
0,2026-01-12 07:39:16.765647,NaT,1170.0,Heerlijke steak bij Coco's Outback in Amsterdam,,draft,0.0,2025-02-21 09:47:19.029007+00:00,NaT,physical,...,None,2026-01-13 06:58:56.212465,2025-02-21 22:36:28.989647,2025-08-17 09:32:16.899729,2025-02-21 09:47:19.029007,5000.0,15.0,5.0,24.0,1.0
1,2026-01-12 07:39:16.908385,NaT,745.0,Outdoor chalkboard and markers to celebrate go...,,inactive,0.0,2025-03-26 08:56:24.776549+00:00,NaT,physical,...,None,2026-01-13 06:58:56.974979,2025-03-26 09:04:29.048260,2025-09-23 17:47:19.407016,2025-03-26 08:56:24.776549,2500.0,7.0,2.0,0.0,1.0
2,2026-01-12 07:39:16.957884,NaT,1081.0,Bite & Bowling,,draft,0.0,2025-02-21 12:14:54.016826+00:00,NaT,physical,...,None,2026-01-13 06:58:57.118680,2025-02-21 13:06:56.734867,2025-03-03 16:50:00.721277,2025-02-21 12:14:54.016826,5000.0,0.0,1.0,0.0,1.0
3,2026-01-12 07:39:17.032326,NaT,1156.0,Deze fles moet jij hebben!!,,live,0.0,2025-07-11 09:08:59.840326+00:00,NaT,physical,...,None,2026-02-14 03:00:18.752922,2025-03-28 20:20:28.174290,2026-01-14 18:08:39.598603,2025-07-11 09:08:59.840326,5000.0,1.0,3.0,31.0,1.0
4,2026-01-12 07:39:17.071050,NaT,1156.0,Review onze 750ml Saywhat Bottle,,live,0.0,2025-07-10 10:39:33.977533+00:00,NaT,physical,...,None,2026-02-02 06:39:14.877789,2025-03-27 19:10:01.782889,2026-01-03 01:58:17.550659,2025-07-10 10:39:33.977533,5000.0,0.0,2.0,13.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7461,2026-02-20 23:02:46.382616,NaT,3506.0,Business Venue Experience at Sterckxhof,"#Sterckxhof,#B2BEvents,#EventVenue,#CorporateE...",finished,NaN,2025-12-16 13:41:54.150654+00:00,2026-02-20 22:59:00+00:00,online,...,None,2026-02-20 23:02:47.278220,2025-12-16 14:38:25.552808,2026-01-14 13:17:16.601495,2025-12-16 13:41:54.150654,5000.0,4.0,1.0,0.0,0.0
7462,2026-02-21 23:02:44.481281,NaT,NaN,mache unsere burritos bekannt!,"#dresden,#burrito,#dresdenfood,#bringyourhasht...",finished,100.0,2026-01-20 07:33:02.358192+00:00,2026-02-21 22:59:00+00:00,physical,...,None,2026-02-21 23:02:45.483654,NaT,NaT,2026-01-20 07:33:02.358192,5000.0,0.0,0.0,0.0,1.0
7463,2026-02-21 23:02:45.490673,NaT,NaN,PANCAKES Amsterdam 1 Parent + 1 Kid Pancake Deal,#PANCAKESAmsterdam,finished,0.0,2026-01-23 11:25:02.946365+00:00,2026-02-21 22:59:00+00:00,physical,...,None,2026-02-21 23:02:46.479121,NaT,NaT,2026-01-23 11:25:02.946365,2500.0,0.0,0.0,0.0,1.0
7464,2026-02-21 23:02:46.579424,NaT,NaN,Hairwonder Long Lasting Haarkleuring - how to,#hairwonder,finished,100.0,2026-01-28 11:20:33.887768+00:00,2026-02-21 22:59:00+00:00,online,...,None,2026-02-21 23:02:47.398936,2026-01-28 16:35:04.200807,2026-01-29 16:43:44.715209,2026-01-28 11:20:33.887768,2500.0,6.0,0.0,0.0,0.0


In [10]:
df_deals.to_parquet('../data/raw/BARTER_DEALS.parquet')